## Summary

Dynamic range quantization was applied to the baseline CNN models for NSL-KDD and UNSW-NB15. The quantized TensorFlow Lite models reduced model size by approximately 90% while maintaining almost identical accuracy, precision, recall, and F1-score. Inference time also improved significantly, supporting the thesis objective of developing a lightweight IDS for resource-constrained IoT devices.

In [22]:
import os

files_to_check = [
    "../models/nslkdd_cnn_quantized_dynamic.tflite",
    "../models/unsw_cnn_quantized_dynamic.tflite",
    "../results/quantization_model_size_results.csv",
    "../results/quantized_tflite_evaluation_results.csv",
    "../results/nslkdd_quantized_tflite_confusion_matrix.csv",
    "../results/unsw_quantized_tflite_confusion_matrix.csv",
    "../results/baseline_vs_quantized_comparison.csv"
]

for file in files_to_check:
    print(file, os.path.exists(file))

../models/nslkdd_cnn_quantized_dynamic.tflite True
../models/unsw_cnn_quantized_dynamic.tflite True
../results/quantization_model_size_results.csv True
../results/quantized_tflite_evaluation_results.csv True
../results/nslkdd_quantized_tflite_confusion_matrix.csv True
../results/unsw_quantized_tflite_confusion_matrix.csv True
../results/baseline_vs_quantized_comparison.csv True


The dynamic range quantized TensorFlow Lite models achieved almost identical detection performance compared with the baseline CNN models. For NSL-KDD, the F1-score changed from 74.94% to 74.95%, while model size decreased from 0.489 MB to 0.046 MB. For UNSW-NB15, the F1-score remained approximately 94.18%, while model size decreased from 0.536 MB to 0.050 MB.

The inference time also improved significantly. On NSL-KDD, inference time decreased from 2.750 seconds to 0.579 seconds. On UNSW-NB15, inference time decreased from 14.661 seconds to 3.677 seconds. These results show that TensorFlow Lite dynamic range quantization is effective for reducing storage and inference cost while preserving intrusion detection performance, making the CNN IDS more suitable for resource-constrained IoT environments.

Quantized TFLite models are not trained from scratch. They are converted from already trained baseline CNN models. Therefore, training time is not applicable for post-training quantization.

In [21]:
thesis_comparison = comparison_results.copy()

for col in ["Accuracy", "Precision", "Recall", "F1-score"]:
    thesis_comparison[col] = (thesis_comparison[col] * 100).round(2)

if "Training Time Seconds" in thesis_comparison.columns:
    thesis_comparison["Training Time Seconds"] = thesis_comparison["Training Time Seconds"].round(2)

thesis_comparison["Inference Time Seconds"] = thesis_comparison["Inference Time Seconds"].round(3)
thesis_comparison["Model Size MB"] = thesis_comparison["Model Size MB"].round(3)

thesis_comparison

,Dataset,Model,Accuracy,Precision,Recall,F1-score,Training Time Seconds,Inference Time Seconds,Model Size MB
0,NSL-KDD,Baseline CNN,76.61,96.02,61.45,74.94,137.07,2.750,0.489
1,UNSW-NB15,Baseline CNN,92.23,96.10,92.33,94.18,43.73,14.661,0.536
2,NSL-KDD,Dynamic Quantized TFLite CNN,76.62,96.04,61.46,74.95,NaN,0.579,0.046
3,UNSW-NB15,Dynamic Quantized TFLite CNN,92.23,96.10,92.33,94.18,NaN,3.677,0.050


In [20]:
BASELINE_COMPARISON_PATH = RESULTS_DIR / "baseline_cnn_comparison.csv"
QUANTIZED_EVALUATION_PATH = RESULTS_DIR / "quantized_tflite_evaluation_results.csv"

baseline_results = pd.read_csv(BASELINE_COMPARISON_PATH)
quantized_results = pd.read_csv(QUANTIZED_EVALUATION_PATH)

comparison_results = pd.concat(
    [baseline_results, quantized_results],
    ignore_index=True
)

FINAL_COMPARISON_PATH = RESULTS_DIR / "baseline_vs_quantized_comparison.csv"
comparison_results.to_csv(FINAL_COMPARISON_PATH, index=False)

print("Final comparison saved to:", FINAL_COMPARISON_PATH)
comparison_results

Final comparison saved to: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\results\baseline_vs_quantized_comparison.csv


,Dataset,Model,Accuracy,Precision,Recall,F1-score,Training Time Seconds,Inference Time Seconds,Model Size MB
0,NSL-KDD,Baseline CNN,0.766057,0.960185,0.614509,0.749406,137.072274,2.750228,0.488823
1,UNSW-NB15,Baseline CNN,0.922289,0.961005,0.923287,0.941769,43.730621,14.660965,0.535698
2,NSL-KDD,Dynamic Quantized TFLite CNN,0.766191,0.960424,0.614587,0.749537,NaN,0.578795,0.046349
3,UNSW-NB15,Dynamic Quantized TFLite CNN,0.922300,0.961038,0.923270,0.941776,NaN,3.677338,0.050255


In [19]:
nslkdd_quantized_cm_path = RESULTS_DIR / "nslkdd_quantized_tflite_confusion_matrix.csv"
unsw_quantized_cm_path = RESULTS_DIR / "unsw_quantized_tflite_confusion_matrix.csv"

pd.DataFrame(
    nslkdd_tflite_cm,
    index=["Actual Normal", "Actual Attack"],
    columns=["Predicted Normal", "Predicted Attack"]
).to_csv(nslkdd_quantized_cm_path)

pd.DataFrame(
    unsw_tflite_cm,
    index=["Actual Normal", "Actual Attack"],
    columns=["Predicted Normal", "Predicted Attack"]
).to_csv(unsw_quantized_cm_path)

print("NSL-KDD quantized confusion matrix saved:", nslkdd_quantized_cm_path.exists())
print("UNSW-NB15 quantized confusion matrix saved:", unsw_quantized_cm_path.exists())

NSL-KDD quantized confusion matrix saved: True
UNSW-NB15 quantized confusion matrix saved: True


In [18]:
quantized_evaluation_results = pd.DataFrame([
    {
        "Dataset": "NSL-KDD",
        "Model": "Dynamic Quantized TFLite CNN",
        "Accuracy": nslkdd_tflite_accuracy,
        "Precision": nslkdd_tflite_precision,
        "Recall": nslkdd_tflite_recall,
        "F1-score": nslkdd_tflite_f1,
        "Inference Time Seconds": nslkdd_tflite_inference_time,
        "Model Size MB": get_file_size_mb(NSLKDD_TFLITE_PATH)
    },
    {
        "Dataset": "UNSW-NB15",
        "Model": "Dynamic Quantized TFLite CNN",
        "Accuracy": unsw_tflite_accuracy,
        "Precision": unsw_tflite_precision,
        "Recall": unsw_tflite_recall,
        "F1-score": unsw_tflite_f1,
        "Inference Time Seconds": unsw_tflite_inference_time,
        "Model Size MB": get_file_size_mb(UNSW_TFLITE_PATH)
    }
])

QUANTIZED_EVALUATION_PATH = RESULTS_DIR / "quantized_tflite_evaluation_results.csv"
quantized_evaluation_results.to_csv(QUANTIZED_EVALUATION_PATH, index=False)

print("Saved to:", QUANTIZED_EVALUATION_PATH)
quantized_evaluation_results

Saved to: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\results\quantized_tflite_evaluation_results.csv


,Dataset,Model,Accuracy,Precision,Recall,F1-score,Inference Time Seconds,Model Size MB
0,NSL-KDD,Dynamic Quantized TFLite CNN,0.766191,0.960424,0.614587,0.749537,0.578795,0.046349
1,UNSW-NB15,Dynamic Quantized TFLite CNN,0.922300,0.961038,0.923270,0.941776,3.677338,0.050255


NSL-KDD:
Baseline F1:   0.749406
Quantized F1:  0.749537
Model size:    0.489 MB -> 0.046 MB
Inference:     2.75 sec -> 0.48 sec

UNSW-NB15:
Baseline F1:   0.941769
Quantized F1:  0.941776
Model size:    0.536 MB -> 0.050 MB
Inference:     14.08 sec -> 4.08 sec

In [17]:
unsw_tflite_pred, unsw_tflite_inference_time = predict_with_tflite(
    UNSW_TFLITE_PATH,
    X_unsw_test_cnn
)

unsw_tflite_accuracy = accuracy_score(y_unsw_test, unsw_tflite_pred)
unsw_tflite_precision = precision_score(y_unsw_test, unsw_tflite_pred, zero_division=0)
unsw_tflite_recall = recall_score(y_unsw_test, unsw_tflite_pred, zero_division=0)
unsw_tflite_f1 = f1_score(y_unsw_test, unsw_tflite_pred, zero_division=0)

print("UNSW-NB15 Dynamic Quantized TFLite Results")
print(f"Accuracy: {unsw_tflite_accuracy:.6f}")
print(f"Precision: {unsw_tflite_precision:.6f}")
print(f"Recall: {unsw_tflite_recall:.6f}")
print(f"F1-score: {unsw_tflite_f1:.6f}")
print(f"Inference time: {unsw_tflite_inference_time:.6f} seconds")

unsw_tflite_cm = confusion_matrix(y_unsw_test, unsw_tflite_pred)

pd.DataFrame(
    unsw_tflite_cm,
    index=["Actual Normal", "Actual Attack"],
    columns=["Predicted Normal", "Predicted Attack"]
)

c:\Users\Admin\Desktop\Lightweight-IoT-IDS\thesis_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


UNSW-NB15 Dynamic Quantized TFLite Results
Accuracy: 0.922300
Precision: 0.961038
Recall: 0.923270
F1-score: 0.941776
Inference time: 3.677338 seconds


,Predicted Normal,Predicted Attack
Actual Normal,51533,4467
Actual Attack,9157,110184


In [16]:
nslkdd_tflite_pred, nslkdd_tflite_inference_time = predict_with_tflite(
    NSLKDD_TFLITE_PATH,
    X_nslkdd_test_cnn
)

nslkdd_tflite_accuracy = accuracy_score(y_nslkdd_test, nslkdd_tflite_pred)
nslkdd_tflite_precision = precision_score(y_nslkdd_test, nslkdd_tflite_pred, zero_division=0)
nslkdd_tflite_recall = recall_score(y_nslkdd_test, nslkdd_tflite_pred, zero_division=0)
nslkdd_tflite_f1 = f1_score(y_nslkdd_test, nslkdd_tflite_pred, zero_division=0)

print("NSL-KDD Dynamic Quantized TFLite Results")
print(f"Accuracy: {nslkdd_tflite_accuracy:.6f}")
print(f"Precision: {nslkdd_tflite_precision:.6f}")
print(f"Recall: {nslkdd_tflite_recall:.6f}")
print(f"F1-score: {nslkdd_tflite_f1:.6f}")
print(f"Inference time: {nslkdd_tflite_inference_time:.6f} seconds")

nslkdd_tflite_cm = confusion_matrix(y_nslkdd_test, nslkdd_tflite_pred)

pd.DataFrame(
    nslkdd_tflite_cm,
    index=["Actual Normal", "Actual Attack"],
    columns=["Predicted Normal", "Predicted Attack"]
)

c:\Users\Admin\Desktop\Lightweight-IoT-IDS\thesis_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


NSL-KDD Dynamic Quantized TFLite Results
Accuracy: 0.766191
Precision: 0.960424
Recall: 0.614587
F1-score: 0.749537
Inference time: 0.578795 seconds


,Predicted Normal,Predicted Attack
Actual Normal,9386,325
Actual Attack,4946,7887


NSL-KDD TFLite input:  (22544, 41, 1)
UNSW-NB15 TFLite input: (175341, 42, 1)

In [15]:
UNSW_TARGET_COLUMN = "label"
UNSW_DROP_COLUMNS = ["label", "attack_cat"]

X_unsw_test = unsw_test_df.drop(columns=UNSW_DROP_COLUMNS)
y_unsw_test = unsw_test_df[UNSW_TARGET_COLUMN]

X_unsw_test_scaled = unsw_scaler.transform(X_unsw_test)
X_unsw_test_cnn = X_unsw_test_scaled.reshape(
    X_unsw_test_scaled.shape[0],
    X_unsw_test_scaled.shape[1],
    1
)

print("UNSW-NB15 CNN test shape:", X_unsw_test_cnn.shape)
print("UNSW-NB15 target distribution:")
print(y_unsw_test.value_counts())

UNSW-NB15 CNN test shape: (175341, 42, 1)
UNSW-NB15 target distribution:
label
1    119341
0     56000
Name: count, dtype: int64


In [14]:
NSLKDD_TARGET_COLUMN = "label"

X_nslkdd_test = nslkdd_test_df.drop(columns=[NSLKDD_TARGET_COLUMN])
y_nslkdd_test = nslkdd_test_df[NSLKDD_TARGET_COLUMN]

X_nslkdd_test_scaled = nslkdd_scaler.transform(X_nslkdd_test)
X_nslkdd_test_cnn = X_nslkdd_test_scaled.reshape(
    X_nslkdd_test_scaled.shape[0],
    X_nslkdd_test_scaled.shape[1],
    1
)

print("NSL-KDD CNN test shape:", X_nslkdd_test_cnn.shape)
print("NSL-KDD target distribution:")
print(y_nslkdd_test.value_counts())

NSL-KDD CNN test shape: (22544, 41, 1)
NSL-KDD target distribution:
label
1    12833
0     9711
Name: count, dtype: int64


In [13]:
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

DATA_DIR = PROJECT_ROOT / "datasets" / "processed"

NSLKDD_TEST_PATH = DATA_DIR / "nslkdd_test_processed.csv"
UNSW_TEST_PATH = DATA_DIR / "unsw_test_processed.csv"

NSLKDD_SCALER_PATH = MODEL_DIR / "nslkdd_scaler.pkl"
UNSW_SCALER_PATH = MODEL_DIR / "unsw_scaler.pkl"

nslkdd_test_df = pd.read_csv(NSLKDD_TEST_PATH)
unsw_test_df = pd.read_csv(UNSW_TEST_PATH)

nslkdd_scaler = joblib.load(NSLKDD_SCALER_PATH)
unsw_scaler = joblib.load(UNSW_SCALER_PATH)

print("NSL-KDD test shape:", nslkdd_test_df.shape)
print("UNSW-NB15 test shape:", unsw_test_df.shape)

NSL-KDD test shape: (22544, 42)
UNSW-NB15 test shape: (175341, 44)


In [12]:
def predict_with_tflite(tflite_model_path, X_cnn):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_model_path))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_index = input_details[0]["index"]
    output_index = output_details[0]["index"]

    predictions = []

    start_time = time.time()

    for i in range(X_cnn.shape[0]):
        sample = X_cnn[i:i+1].astype(np.float32)
        interpreter.set_tensor(input_index, sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_index)
        predictions.append(output[0][0])

    inference_time = time.time() - start_time

    predictions = np.array(predictions)
    predicted_labels = (predictions >= 0.5).astype(int)

    return predicted_labels, inference_time

NSL-KDD:
0.489 MB -> 0.046 MB
Size reduction: 90.52%

UNSW-NB15:
0.536 MB -> 0.050 MB
Size reduction: 90.62%

In [11]:
print("NSL-KDD quantized model exists:", NSLKDD_TFLITE_PATH.exists())
print("UNSW-NB15 quantized model exists:", UNSW_TFLITE_PATH.exists())
print("Quantization results exist:", QUANTIZATION_RESULTS_PATH.exists())

print("\nSaved results:")
print(pd.read_csv(QUANTIZATION_RESULTS_PATH))

NSL-KDD quantized model exists: True
UNSW-NB15 quantized model exists: True
Quantization results exist: True

Saved results:
     Dataset Baseline Model                 Optimized Model  Baseline Size MB  \
0    NSL-KDD        CNN .h5  Dynamic Range Quantized TFLite          0.488823   
1  UNSW-NB15        CNN .h5  Dynamic Range Quantized TFLite          0.535698   

   Quantized Size MB  Size Reduction MB  Size Reduction Percent  
0           0.046349           0.442474               90.518331  
1           0.050255           0.485443               90.618814  


In [9]:
size_results = pd.DataFrame([
    {
        "Dataset": "NSL-KDD",
        "Baseline Model": "CNN .h5",
        "Optimized Model": "Dynamic Range Quantized TFLite",
        "Baseline Size MB": get_file_size_mb(NSLKDD_MODEL_PATH),
        "Quantized Size MB": get_file_size_mb(NSLKDD_TFLITE_PATH),
    },
    {
        "Dataset": "UNSW-NB15",
        "Baseline Model": "CNN .h5",
        "Optimized Model": "Dynamic Range Quantized TFLite",
        "Baseline Size MB": get_file_size_mb(UNSW_MODEL_PATH),
        "Quantized Size MB": get_file_size_mb(UNSW_TFLITE_PATH),
    }
])

size_results["Size Reduction MB"] = (
    size_results["Baseline Size MB"] - size_results["Quantized Size MB"]
)

size_results["Size Reduction Percent"] = (
    size_results["Size Reduction MB"] / size_results["Baseline Size MB"] * 100
)

size_results.to_csv(QUANTIZATION_RESULTS_PATH, index=False)

size_results

,Dataset,Baseline Model,Optimized Model,Baseline Size MB,Quantized Size MB,Size Reduction MB,Size Reduction Percent
0,NSL-KDD,CNN .h5,Dynamic Range Quantized TFLite,0.488823,0.046349,0.442474,90.518331
1,UNSW-NB15,CNN .h5,Dynamic Range Quantized TFLite,0.535698,0.050255,0.485443,90.618814


In [7]:
convert_to_dynamic_quantized_tflite(NSLKDD_MODEL_PATH, NSLKDD_TFLITE_PATH)
convert_to_dynamic_quantized_tflite(UNSW_MODEL_PATH, UNSW_TFLITE_PATH)

print("NSL-KDD quantized model saved:", NSLKDD_TFLITE_PATH)
print("UNSW-NB15 quantized model saved:", UNSW_TFLITE_PATH)

print("NSL-KDD TFLite exists:", NSLKDD_TFLITE_PATH.exists())
print("UNSW-NB15 TFLite exists:", UNSW_TFLITE_PATH.exists())

INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpjepj_wms\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpjepj_wms\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpjepj_wms'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 41, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1887398146704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887398148240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887398148816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887399248912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887399248336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887399250256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887399249296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887399251600: TensorSpec(shape=(), dtype=tf.resource, name=None)


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpy0f39lcb\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpy0f39lcb\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpy0f39lcb'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 42, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1887399256400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887399251984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887399259280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887399259088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887399254096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887399255440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887399259664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1887399256208: TensorSpec(shape=(), dtype=tf.resource, name=None)
NSL-KDD quantized model saved: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\models\nslkdd_cnn_quantized_dynamic.tflite
UNSW-NB15 quantized model saved: 

In [6]:
def convert_to_dynamic_quantized_tflite(keras_model_path, output_tflite_path):
    model = tf.keras.models.load_model(keras_model_path)

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    tflite_model = converter.convert()

    output_tflite_path.write_bytes(tflite_model)

    return output_tflite_path

In [5]:
def get_file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 * 1024)

print("NSL-KDD baseline model size MB:", get_file_size_mb(NSLKDD_MODEL_PATH))
print("UNSW-NB15 baseline model size MB:", get_file_size_mb(UNSW_MODEL_PATH))

NSL-KDD baseline model size MB: 0.48882293701171875
UNSW-NB15 baseline model size MB: 0.5356979370117188


In [1]:
from pathlib import Path
import os
import time

import numpy as np
import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path("..").resolve()

MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

NSLKDD_MODEL_PATH = MODEL_DIR / "nslkdd_cnn_baseline_model.h5"
UNSW_MODEL_PATH = MODEL_DIR / "unsw_cnn_baseline_model.h5"

NSLKDD_TFLITE_PATH = MODEL_DIR / "nslkdd_cnn_quantized_dynamic.tflite"
UNSW_TFLITE_PATH = MODEL_DIR / "unsw_cnn_quantized_dynamic.tflite"

QUANTIZATION_RESULTS_PATH = RESULTS_DIR / "quantization_model_size_results.csv"

print("TensorFlow version:", tf.__version__)
print("NSL-KDD model exists:", NSLKDD_MODEL_PATH.exists())
print("UNSW-NB15 model exists:", UNSW_MODEL_PATH.exists())

TensorFlow version: 2.21.0
NSL-KDD model exists: True
UNSW-NB15 model exists: True
